# Leaving Limbo — Syria's Displacement Crisis 2024–2025
**Leaflet · D3 · Canvas · IntersectionObserver · vanilla JS**

*Syria · December 2024 – December 2025 · 8,325 communities monitored*

This notebook documents every layer of `syria_story.html` — a single-file scrollytelling story.
Python cells are executable. HTML / CSS / JS blocks are annotated reference code drawn directly
from the published file.

## Story Overview

When the Assad government fell on **December 8, 2024**, roughly two million Syrians began returning
home. Six million remain internally displaced. A parallel story — largely uncovered — ran at the
same time: al-Hol, the ISIS-associated detention camp in northeastern Syria, began emptying in
January 2026 with no coordinated reintegration plan.

**Structure:**
- Hero: animated dot canvas of all 8,325 monitored communities
- 10 narrative panels, each paired with a map flyTo state
- 4 full-bleed video sections (Al Jazeera + WION) with distinct layouts
- D3 v7 animated bar chart of monthly returnee flows
- Leaflet dark map with a custom canvas dot layer
- Delivered as a single 1,700-line HTML file — no build step, no framework

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display

# Match the story's dark palette for all plots
plt.rcParams.update({
    'figure.facecolor': '#0b0d12',
    'axes.facecolor':   '#0b0d12',
    'axes.edgecolor':   '#1e293b',
    'text.color':       '#94a3b8',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#475569',
    'ytick.color':      '#475569',
    'font.family':      'sans-serif',
    'font.size':        11,
})

## 1 · Data

The underlying dataset is UNHCR/OCHA Displacement Tracking Matrix for Syria, December 2025.
**8,325 community records** across all 14 governorates, inlined as a JSON array directly in the
HTML — no API call at runtime.

| Field | Type | Description |
|-------|------|-------------|
| `la` | float | Latitude |
| `ln` | float | Longitude |
| `i`  | int | IDP population count |
| `r`  | int | Returnees since December 2024 |
| `g`  | string | Governorate |
| `n`  | string | Community name |
| `d`  | string | District |

The full array is ~800 KB inlined. The browser handles it via a `Float32Array` style cache
(covered in §4) so parsing happens once at load.

In [ ]:
# Representative sample — the full 8,325-record array is embedded in syria_story.html
sample = [
    {'la': 35.198, 'ln': 35.963, 'i': 500,    'r': 6402,  'g': 'Tartous',     'n': 'Banyas',             'd': 'Banyas'},
    {'la': 36.622, 'ln': 39.326, 'i': 18,     'r': 0,     'g': 'Ar-Raqqa',    'n': 'Kutoum',             'd': 'Tell Abiad'},
    {'la': 32.754, 'ln': 36.131, 'i': 638,    'r': 0,     'g': "Dar'a",       'n': 'Dael',               'd': "Dar'a"},
    {'la': 33.488, 'ln': 36.348, 'i': 501000, 'r': 0,     'g': 'Rif Dimashq', 'n': 'Jaramana',           'd': 'Jaramana'},
    {'la': 33.436, 'ln': 36.245, 'i': 240000, 'r': 0,     'g': 'Rif Dimashq', 'n': 'Ashrafiet Sahnaya',  'd': 'Jaramana'},
    {'la': 36.237, 'ln': 37.153, 'i': 270000, 'r': 0,     'g': 'Aleppo',      'n': 'Ash-Sheikh Maqsoud', 'd': 'Jebel Saman'},
    {'la': 36.585, 'ln': 37.043, 'i': 142000, 'r': 0,     'g': 'Aleppo',      'n': 'Azaz',               'd': "A'zaz"},
    {'la': 35.648, 'ln': 36.677, 'i': 0,      'r': 36000, 'g': 'Idleb',       'n': 'Maarrat An-Numan',   'd': 'Maarrat An-Numan'},
    {'la': 34.924, 'ln': 36.731, 'i': 0,      'r': 7370,  'g': 'Homs',        'n': 'Ar-Rastan',          'd': 'Ar-Rastan'},
    {'la': 35.414, 'ln': 36.389, 'i': 0,      'r': 20000, 'g': 'Hama',        'n': 'Madiq Castle',       'd': 'Madiq Castle'},
]

df = pd.DataFrame(sample)
df.columns = ['lat', 'lon', 'idps', 'returnees', 'governorate', 'name', 'district']
display(df[['name', 'governorate', 'district', 'idps', 'returnees']])

In [ ]:
# Governorate-level aggregates (from story highlight data)
gov = {
    'Aleppo':       (1_280_000, 506_000),
    'Rif Dimashq':  (1_920_000, 142_000),
    'Idleb':        (  958_000, 504_000),
    'Hama':         (  420_000, 344_000),
    'Homs':         (  380_000, 198_000),
    'Deir ez-Zor':  (  290_000,  88_000),
    "Dar'a":        (  210_000,  76_000),
    'Tartous':      (  180_000,  28_000),
}

names = list(gov.keys())
idps  = [gov[k][0] / 1e6 for k in names]
rets  = [gov[k][1] / 1e6 for k in names]
y     = np.arange(len(names))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(y - 0.2, idps, height=0.35, color='#38bdf8', alpha=0.82, label='IDPs')
ax.barh(y + 0.2, rets, height=0.35, color='#f59e0b', alpha=0.82, label='Returnees')
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.set_xlabel('Population (millions)')
ax.set_title('IDPs vs. returnees by governorate — December 2025', color='#e2e8f0', pad=12)
ax.legend(framealpha=0.1, edgecolor='#334155')
ax.yaxis.grid(True, alpha=0.12)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 2 · Monthly Return Flow

`MONTHLY` tracks returnee counts by month from December 2024 through December 2025.
The **January 2025 spike — 380,651 people in a single month** — captures the immediate
post-Assad surge: families who had waited years moved the moment checkpoints dissolved.
The trend afterward shows sustained but decelerating return.

The D3 chart in the story animates each bar sequentially with a 65 ms stagger and highlights
the January peak with a brighter color and a value label.

In [ ]:
# MONTHLY data — mirrors the JS constant in syria_story.html
monthly = [
    ('Dec 2024', 189183), ('Jan 2025', 380651), ('Feb 2025', 114816),
    ('Mar 2025', 120148), ('Apr 2025', 163940), ('May 2025', 186780),
    ('Jun 2025', 149851), ('Jul 2025', 133998), ('Aug 2025', 116441),
    ('Nov 2025',  52459), ('Dec 2025',  72141),
]

months, counts = zip(*monthly)
colors = ['#fcd34d' if m == 'Jan 2025' else '#f59e0b' for m in months]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(months, [c / 1000 for c in counts],
              color=colors, alpha=0.85, width=0.65, zorder=3)
ax.bar_label(bars,
    labels=['380K' if m == 'Jan 2025' else '' for m in months],
    padding=4, color='#fcd34d', fontsize=10, fontweight='bold')
ax.set_ylabel('Returnees (thousands)')
ax.set_title('Monthly Syrian returnees — Dec 2024 to Dec 2025', color='#e2e8f0', pad=12)
ax.tick_params(axis='x', rotation=40)
ax.yaxis.grid(True, alpha=0.15, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# TOTALS — mirrors the JS constant in syria_story.html
totals = {
    'returnees_internal': 1_911_262,
    'idps_remaining':     5_963_668,
    'arrivals_abroad':      583_303,
    'arrivals_origin':      516_842,
    'arrivals_other':        66_461,
}

ratio    = totals['idps_remaining'] / totals['returnees_internal']
pct_orig = totals['arrivals_origin'] / totals['arrivals_abroad'] * 100

print('Internal returnees:         {:>12,}'.format(totals['returnees_internal']))
print('Still internally displaced: {:>12,}'.format(totals['idps_remaining']))
print('IDP-to-returnee ratio:      {:>12.1f} : 1'.format(ratio))
print()
print('Returned from abroad:       {:>12,}'.format(totals['arrivals_abroad']))
print('  to origin community:      {:>12,}  ({:.0f}%)'.format(totals['arrivals_origin'], pct_orig))
print('  to different location:    {:>12,}  ({:.0f}%)'.format(totals['arrivals_other'], 100 - pct_orig))

## 3 · Style Engine

Five visual states — `neutral`, `idp`, `ret`, `both`, `clear` — each map a location's
data to `[R, G, B, alpha, radius]`. Two color scales:

- **Blue** (`#1d4ed8` → `#93c5fd`): IDP density — darker and larger = more displaced
- **Amber** (`#b45309` → `#fde68a`): returnee count — darker and larger = more came back

```javascript
const SD = {
  neutral: d => [51, 65, 85, 0.45, 2],

  idp: d => {
    if (d.i > 200000) return [29,  78, 216, 0.95, 14];
    if (d.i >  50000) return [37,  99, 235, 0.88, 11];
    if (d.i >  10000) return [59, 130, 246, 0.78,  8];
    if (d.i >   2000) return [96, 165, 250, 0.65,  5];
    if (d.i >      0) return [147, 197, 253, 0.50, 2.5];
    return [30, 41, 59, 0.18, 1.8];
  },

  ret: d => {
    if (d.r > 30000) return [180,  83,  9, 0.95, 12];
    if (d.r > 10000) return [217, 119,  6, 0.88,  9];
    if (d.r >  3000) return [245, 158, 11, 0.80,  6];
    if (d.r >   500) return [251, 191, 36, 0.68,  4];
    if (d.r >     0) return [252, 211, 77, 0.50, 2.5];
    return [30, 41, 59, 0.18, 1.8];
  },
};
```

In [ ]:
# Python translation of SD.idp and SD.ret — same thresholds as the JS
def style_idp(n):
    if n > 200_000: return (29,  78, 216, 0.95, 14)
    if n >  50_000: return (37,  99, 235, 0.88, 11)
    if n >  10_000: return (59, 130, 246, 0.78,  8)
    if n >   2_000: return (96, 165, 250, 0.65,  5)
    if n >       0: return (147, 197, 253, 0.50, 2.5)
    return (30, 41, 59, 0.18, 1.8)

def style_ret(n):
    if n > 30_000: return (180,  83,  9, 0.95, 12)
    if n > 10_000: return (217, 119,  6, 0.88,  9)
    if n >  3_000: return (245, 158, 11, 0.80,  6)
    if n >    500: return (251, 191, 36, 0.68,  4)
    if n >      0: return (252, 211, 77, 0.50, 2.5)
    return (30, 41, 59, 0.18, 1.8)

In [ ]:
# Visualize the two color scales as dot swatches
buckets_idp = [(0,'none'), (1,'1–2K'), (2001,'2–10K'), (10001,'10–50K'), (50001,'50–200K'), (200001,'200K+')]
buckets_ret = [(0,'none'), (1,'1–500'), (501,'500–3K'), (3001,'3–10K'), (10001,'10–30K'), (30001,'30K+')]

fig, axes = plt.subplots(1, 2, figsize=(10, 1.9))
for ax, buckets, style_fn, title in [
    (axes[0], buckets_idp, style_idp, 'IDP scale (blue)'),
    (axes[1], buckets_ret, style_ret, 'Returnee scale (amber)'),
]:
    for j, (threshold, label) in enumerate(buckets):
        r, g, b, a, rad = style_fn(threshold)
        ax.add_patch(mpatches.FancyBboxPatch(
            (j, 0.1), 0.82, 0.82, boxstyle='round,pad=0.05',
            facecolor=(r/255, g/255, b/255, a)))
        ax.text(j + 0.41, -0.08, label, ha='center', va='top', fontsize=8, color='#64748b')
    ax.set_xlim(-0.1, len(buckets))
    ax.set_ylim(-0.45, 1.1)
    ax.axis('off')
    ax.set_title(title, color='#94a3b8', fontsize=10, pad=6)
plt.tight_layout()
plt.show()

## 4 · Float32Array Precomputation

At startup the style engine iterates once over all 8,325 locations and stores results for each
of the five states in a typed array. Each location needs 5 values, so each buffer is
`N × 5 = 41,625` floats. Transitions interpolate `curr[]` between `from[]` and `to[]`
in the rAF loop — no per-frame style recomputation.

```javascript
const N  = ALL_LOCS.length;   // 8,325
const SA = {};
['neutral', 'idp', 'ret', 'both', 'clear'].forEach(state => {
  const a = new Float32Array(N * 5);
  for (let i = 0; i < N; i++) {
    const [r, g, b, al, rd] = SD[state](ALL_LOCS[i]);
    a[i*5]=r; a[i*5+1]=g; a[i*5+2]=b; a[i*5+3]=al*255; a[i*5+4]=rd;
  }
  SA[state] = a;
});

let curr = SA['neutral'].slice(), from = curr, to = null, tS = 0;
const TD = 550;   // transition duration in ms

function startT(state) { from = curr.slice(); to = SA[state]; tS = performance.now(); }
function eio(t) { return t < .5 ? 2*t*t : -1 + (4 - 2*t) * t; }   // ease-in-out
```

In [ ]:
# Visualize the ease-in-out curve used for all style transitions
t = np.linspace(0, 1, 300)
eio = np.where(t < 0.5, 2*t**2, -1 + (4 - 2*t)*t)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, eio, color='#f59e0b', lw=2.5, label='ease-in-out  (550 ms)')
ax.plot(t, t,   color='#475569', lw=1,   ls='--', label='linear (no easing)')
ax.fill_between(t, t, eio, where=(eio > t), alpha=0.08, color='#f59e0b')
ax.set_xlabel('raw progress  (ts - tStart) / 550')
ax.set_ylabel('eased progress')
ax.set_title('Style transition easing curve', color='#e2e8f0', pad=10)
ax.legend(framealpha=0.1, edgecolor='#334155')
ax.yaxis.grid(True, alpha=0.12)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 5 · Map Initialization

Leaflet 1.9.4 on a CARTO dark tile layer. `scrollWheelZoom: false` is the critical fix —
without it, Leaflet intercepts the mouse wheel and the page never scrolls.

```javascript
const map = L.map('map', {
  center: [34.8, 38.5],
  zoom: 6,
  zoomControl: false,
  minZoom: 5, maxZoom: 16,
  scrollWheelZoom: false,    // prevents map capturing page scroll
  doubleClickZoom: true,
});

L.tileLayer(
  'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
  { subdomains: 'abcd', maxZoom: 19 }
).addTo(map);
```

Each narrative step calls `map.flyTo([lat, lng], zoom, { duration })` for smooth pan+zoom.
The canvas dot layer repaints every frame by calling `map.latLngToContainerPoint()` to
convert geographic coordinates to pixel positions.

A proximity tooltip fires on `map.on('mousemove')` — scanning a prefiltered `hitL` list
(locations with > 50 IDPs or > 50 returnees) for the nearest point within a 16 px radius.

```javascript
const hitL = ALL_LOCS.filter(d => d.i > 50 || d.r > 50);

map.on('mousemove', e => {
  const { x, y } = e.containerPoint;
  let best = null, bD2 = 16 * 16;
  for (const d of hitL) {
    const pt = map.latLngToContainerPoint([d.la, d.ln]);
    const d2 = (pt.x-x)**2 + (pt.y-y)**2;
    if (d2 < bD2) { bD2 = d2; best = d; }
  }
  if (best) showTooltip(best, e.originalEvent);
  else      tip.style.display = 'none';
});
```

## 6 · Canvas Dot Layer

Leaflet's default SVG/DOM markers are too slow for 8,325 points. The story uses a raw `<canvas>`
absolutely positioned over the map container, repainted every rAF frame.

**Key optimization — batch by visual key.** Instead of calling `beginPath` and `fill` per dot,
the draw loop groups dots with identical (R, G, B, alpha, radius) into one `beginPath` / `fill`
call. This reduces fill calls from 8,325 to roughly 40–60 per frame.

```javascript
function drawDots(ts) {
  // Interpolate curr[] between from[] and to[] using eased progress
  if (to) {
    const raw = Math.min(1, (ts - tS) / TD);
    const t   = eio(raw);
    for (let i = 0; i < N * 5; i++) curr[i] = from[i] + (to[i] - from[i]) * t;
    if (raw >= 1) { curr = to.slice(); to = null; }
  }

  ctx.clearRect(0, 0, cv.width, cv.height);

  // Group dots by visual key → one beginPath / fill per group
  const groups = new Map();
  for (let i = 0; i < N; i++) {
    const b  = i * 5;
    const R  = Math.round(curr[b]),   G = Math.round(curr[b+1]), B = Math.round(curr[b+2]);
    const A  = curr[b+3] / 255;
    const rd = Math.round(curr[b+4] * 2) / 2;   // quantize to 0.5 px steps
    if (A < 0.05 || rd < 0.5) continue;          // skip invisible dots
    const key = `${R},${G},${B},${Math.round(A*100)},${rd}`;
    if (!groups.has(key)) groups.set(key, { R, G, B, A, rad: rd, idx: [] });
    groups.get(key).idx.push(i);
  }

  for (const g of groups.values()) {
    ctx.globalAlpha = g.A;
    ctx.fillStyle   = `rgb(${g.R},${g.G},${g.B})`;
    ctx.beginPath();
    for (const i of g.idx) {
      const d  = ALL_LOCS[i];
      const pt = map.latLngToContainerPoint([d.la, d.ln]);
      ctx.moveTo(pt.x + g.rad, pt.y);
      ctx.arc(pt.x, pt.y, g.rad, 0, Math.PI * 2);
    }
    ctx.fill();
  }
}

function loop(ts) { drawDots(ts); requestAnimationFrame(loop); }
requestAnimationFrame(loop);
```

## 7 · Scrollytelling Layout

The scaffold is a CSS Grid overlap: `.sticky-wrap` and `.content` share `grid-area: 1/1`.
The map sticks while content scrolls over it. Zero JavaScript involved.

```css
.scrolly     { display: grid; grid-template-columns: 100%; }

.sticky-wrap {
  grid-area: 1/1;
  position: sticky; top: 0; height: 100vh;
  align-self: start; z-index: 0;
}

.content {
  grid-area: 1/1;         /* same cell — layered over sticky-wrap */
  position: relative; z-index: 5;
  pointer-events: none;   /* transparent by default */
}
```

Three element types inside `.content`:

| Class | Height | Background | Role |
|-------|--------|------------|------|
| `.window` | 100vh | transparent | Triggers `flyTo` + dot state via IntersectionObserver |
| `.tpanel` | ≥100vh | transparent | Text column 46% wide — map dots visible on right 54% |
| `.vid-section` | 100vh | `#000` | Full-bleed video — covers map entirely |

Text panels fade in from the left on scroll via a CSS transition triggered by IntersectionObserver:

```css
.ti {
  width: min(46%, 610px);
  background: rgba(11, 13, 18, 0.93);
  backdrop-filter: blur(10px);
  opacity: 0; transform: translateX(-24px);
  transition: opacity .55s ease, transform .55s ease;
}
.ti.on { opacity: 1; transform: translateX(0); }
```

The observer fires at a **15% threshold** — early enough that the panel is already animating
in as the reader's attention reaches it.

## 8 · Step Machine

`STEPS[i]` fires when `.window[data-step=i]` enters 50% of the viewport. Each function calls
`map.flyTo()`, `startT()` (dot color transition), `clearHL()`, and optionally `addHL()` for
pulsing location markers.

```javascript
const STEPS = [
  // 0 — neutral overview
  () => { map.flyTo([34.8, 38.5], 6,   { duration: 1.3 }); startT('neutral'); clearHL(); },

  // 1 — full Syria, all 14 governorates
  () => { map.flyTo([34.8, 38.5], 6,   { duration: 0.5 }); startT('neutral'); clearHL(); },

  // 2 — IDP mode: blue dots across Syria
  () => { map.flyTo([34.8, 38.5], 6.5, { duration: 1.1 }); startT('idp'); leg.classList.add('on'); },

  // 3 — zoom to Rural Damascus, highlight 4 IDP hubs
  () => {
    map.flyTo([33.50, 36.26], 11, { duration: 1.9 }); startT('idp'); clearHL();
    addHL(33.48766, 36.34832, 'Jaramana 501K',          '#38bdf8');
    addHL(33.43635, 36.24464, 'Ashrafiet Sahnaya 240K', '#38bdf8');
    addHL(33.60416, 36.31183, 'At-Tall 192K',           '#60a5fa');
    addHL(33.43353, 36.16273, 'Jdidet Artuz 170K',      '#7dd3fc');
  },

  // 4 — zoom to Aleppo
  () => {
    map.flyTo([36.38, 37.22], 10.5, { duration: 1.9 }); startT('idp'); clearHL();
    addHL(36.23661, 37.15285, 'Ash-Sheikh Maqsoud 270K', '#38bdf8');
    addHL(36.58487, 37.04316, 'Azaz 142K',               '#60a5fa');
    addHL(36.36972, 37.51492, 'Al-Bab 139K',             '#7dd3fc');
  },

  // 5 — returnee mode, zoom to Idleb
  () => {
    map.flyTo([35.65, 36.72], 11.5, { duration: 1.9 }); startT('ret'); clearHL();
    addHL(35.64757, 36.67659, 'Maarrat An-Numan 36K', '#f59e0b');
    addHL(35.63534, 36.63312, 'Kafruma 16K',          '#fbbf24');
  },

  // 6 — zoom to Hama
  () => {
    map.flyTo([35.32, 36.53], 11, { duration: 1.9 }); startT('ret'); clearHL();
    addHL(35.41439, 36.38895, 'Madiq Castle 20K', '#f59e0b');
    addHL(35.25851, 36.60896, 'Halfaya 20K',      '#fbbf24');
  },

  // 7, 8, 9 — pull back to full Syria + chart build + final clear
];
```

Two separate `IntersectionObserver` instances keep concerns isolated:

```javascript
// Observer 1 — .window elements at 50% → fire STEPS[], update location label
const ioMap = new IntersectionObserver(entries => {
  entries.forEach(e => {
    if (e.isIntersecting) {
      const i = +e.target.dataset.step;
      pending = i;
      requestAnimationFrame(() => { if (pending === i) STEPS[i]?.(); });
    }
  });
}, { threshold: 0.5 });

// Observer 2 — .ti elements at 15% → toggle fade-in class
const ioText = new IntersectionObserver(entries => {
  entries.forEach(e => e.target.classList.toggle('on', e.isIntersecting));
}, { threshold: 0.15 });
```

The rAF batch in Observer 1 prevents duplicate step triggers when the user scrolls fast enough
to enter and exit a window element within a single frame.

## 9 · Narrative Arc

Ten panels, each paired with a map state and a geographic focus:

| Panel | Map state | Focus | Story beat |
|-------|-----------|-------|------------|
| T1 | neutral | Syria overview | Dec 8, 2024 — Assad falls. Al-Hol frozen in place. |
| T2 | neutral | Full Syria | 8,325 monitored communities — the fractured landscape |
| T3 | idp | Full Syria | 5.96M still displaced. US aid down 58%. |
| T4 | idp zoom | Rural Damascus | Host communities refuse to receive displaced families |
| T5 | idp zoom | Aleppo | 1.28M IDPs and 506K returnees — simultaneously |
| T6 | ret | Southern Idleb | 504K returned. What is home now? |
| T7 | ret | Hama | Hope Convoys — civil society fills the gap |
| T8 | ret | Full Syria | 583K from abroad. Eva Dumani made it. 34 Australians didn't. |
| T9 | ret | Pull back | Monthly chart. Ibrahim Darwish quote. |
| T10 | clear | Full Syria | For every returnee, three remain displaced. |

Four full-bleed video sections interrupt the map at narrative turning points:
after T1, after T2, after T3, and after T8.

## 10 · Video Sections

Four full-bleed `<video>` sections break the scroll rhythm with raw footage from Al Jazeera
and WION. Each uses a distinct overlay layout — same element structure, different visual logic.

---

### Layout 1 — `vo-bottom` · Bottom-left, rising gradient
The WaPo style. Dark gradient climbs from the floor. Text anchors at bottom-left.
Used for the Al Jazeera women-and-children scene (most grounded shot).

```css
.vo-bottom {
  justify-content: flex-end;
  padding: 4rem 3.5rem;
  background: linear-gradient(to top,
    rgba(11,13,18,.95) 0%, rgba(11,13,18,.35) 52%, transparent 82%);
}
```

---

### Layout 2 — `vo-top` · Top-anchor with large stat
Text pinned top-left, leaving most of the frame open for the WION aerial drone footage.
A large typographic number — `75,000` — anchors the upper section.

```css
.vo-top {
  justify-content: flex-start;
  background: linear-gradient(to bottom,
    rgba(11,13,18,.9) 0%, rgba(11,13,18,.25) 42%, transparent 68%);
}
.vo-top .vid-stat {
  font-size: clamp(4rem, 9vw, 7rem);
  font-weight: 700;
  letter-spacing: -.03em;
}
```

---

### Layout 3 — `vo-split` · Center-split, diagonal gradient
Gradient runs left→right (dark on left, open on right). Text centered vertically,
constrained to the left 46% of the frame. Used for the Al Jazeera Akhtarin aerial.

```css
.vo-split {
  justify-content: center;
  background: linear-gradient(105deg,
    rgba(11,13,18,.94) 0%, rgba(11,13,18,.72) 44%, transparent 66%);
}
.vo-split .vid-head { max-width: 46vw; }
.vo-split hr { border-top: 1px solid rgba(255,255,255,.15); width: 3rem; }
```

---

### Layout 4 — `vo-center` · Full-center, radial vignette
Single italic headline centered in the frame. No description text. A ghost quotation-mark
glyph sits decoratively behind it. Video desaturated and darkened to a near-silhouette.
Used for the most visually stark shot: fence + armed guard.

```css
.vo-center {
  justify-content: center; align-items: center; text-align: center;
  background: radial-gradient(ellipse 70% 60% at 50% 50%,
    transparent 25%, rgba(11,13,18,.88) 100%);
}
.vo-center::before {
  content: '"';
  font-size: 18rem;
  color: rgba(255,255,255,.04);
  position: absolute; pointer-events: none;
}
.vo-center .vid-head { font-style: italic; }
```

---

All four videos autoplay and pause with the viewport via `IntersectionObserver`:

```javascript
const ioVid = new IntersectionObserver(entries => {
  entries.forEach(e => {
    if (e.isIntersecting) e.target.play().catch(() => {});
    else                   e.target.pause();
  });
}, { threshold: 0.3 });
document.querySelectorAll('.vid-bg').forEach(el => ioVid.observe(el));
```

HTML media fragments target specific timestamps — three sections reuse the same file:

```html
<source src='videoplayback (1).mp4#t=30'   type='video/mp4'>  <!-- women at shelter -->
<source src='videoplayback.mp4#t=20'       type='video/mp4'>  <!-- WION aerial     -->
<source src='videoplayback (1).mp4#t=0,28' type='video/mp4'>  <!-- Akhtarin buses  -->
<source src='videoplayback (1).mp4#t=150'  type='video/mp4'>  <!-- fence + guard   -->
```

## 11 · Hero Canvas

The hero section uses a second `<canvas>` (separate from the map canvas) that animates the first
900 `ALL_LOCS` entries into a miniature map of Syria using a simple linear projection.
Dots fade in sequentially by index, creating a slow geographic reveal.

```javascript
// Linear lat/lng → pixel — bounding box covers Syria extents
function ll2(lat, lng) {
  const asp = (42.5 - 35.6) / (37.4 - 32.2);   // aspect ratio of the bounding box
  let mw = HW * 0.86, mh = HH * 0.82;
  if (mw / mh > asp) mw = mh * asp; else mh = mw / asp;
  const ox = (HW - mw) / 2, oy = (HH - mh) / 2;
  return [
    ox + (lng - 35.6) / (42.5 - 35.6) * mw,
    oy + (1 - (lat - 32.2) / (37.4 - 32.2)) * mh,
  ];
}

pts = ALL_LOCS.slice(0, 900).map((l, i) => ({
  x: ll2(l.la, l.ln)[0], y: ll2(l.la, l.ln)[1],
  a: 0,                   // alpha starts at 0
  d: i * 2.2,             // staggered delay in frames
  s: 0.014 + Math.random() * 0.014,          // per-dot fade speed
  c: l.i > 0 ? '56,189,248' : '245,158,11', // blue = IDP, amber = returnee
}));
```

The dots appear in the order the dataset was collected (roughly geographic), so the outline
of Syria assembles point by point across the hero screen.

## Summary

| Component | Implementation | Why |
|-----------|---------------|-----|
| Dot layer | Canvas + `Float32Array` | 8,325 points — SVG/DOM too slow |
| Style transitions | Typed-array lerp, 550 ms | Zero GC pressure during crossfade |
| Scroll triggers | `IntersectionObserver` (×2) | No `scroll` listener; no layout thrash |
| Map | Leaflet + `flyTo()` | Built-in smooth pan+zoom |
| Chart | D3 v7 animated bars | Fires once on first viewport entry |
| Video | `<video muted loop playsinline>` + IO | Native autoplay, no library |
| Layout | CSS Grid overlap + `position: sticky` | Zero JS for scrolly scaffold |
| Delivery | Single HTML file | No build step, no framework |